<a href="https://colab.research.google.com/github/Pallavi20004/Demo/blob/main/ID3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter


In [ ]:
def entropy(y):
    class_counts = Counter(y)
    total_samples = len(y)
    return -sum((count / total_samples) * np.log2(count / total_samples) for count in class_counts.values())


In [ ]:
def information_gain(X, y, feature_index):
    unique_values = set(X[:, feature_index])
    total_entropy = entropy(y)
    weighted_entropy = sum((len(y[X[:, feature_index] == value]) / len(y)) * entropy(y[X[:, feature_index] == value]) for value in unique_values)
    return total_entropy - weighted_entropy


In [ ]:
def id3(X, y, features):
    if len(set(y)) == 1:
        return y[0]

    if not features:
        return Counter(y).most_common(1)[0][0]

    best_feature = max(features, key=lambda f: information_gain(X, y, f))

    tree = {best_feature: {}}
    unique_values = set(X[:, best_feature])

    for value in unique_values:
        sub_X = X[X[:, best_feature] == value]
        sub_y = y[X[:, best_feature] == value]
        remaining_features = [f for f in features if f != best_feature]
        tree[best_feature][value] = id3(sub_X, sub_y, remaining_features)

    return tree


In [ ]:
data = pd.DataFrame({
    'Outlook': [0, 0, 1, 2, 2, 2, 1, 0, 0, 2, 0, 1, 1, 2],
    'Temperature': [0, 0, 0, 1, 2, 2, 2, 1, 2, 1, 1, 1, 0, 1],
    'Humidity': [0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1],
    'Wind': [0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1],
    'PlayTennis': [0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0]
})


In [ ]:
X = data.iloc[:, :-1].values  # Features
y = data.iloc[:, -1].values   # Target
features = list(range(X.shape[1]))  # Feature indices

tree = id3(X, y, features)
print("Decision Tree:", tree)


Decision Tree: {0: {0: {1: {0: 0, 1: {2: {0: 0, 1: 1}}, 2: 1}}, 1: 1, 2: {2: {0: 1, 1: 0}}}}


In [ ]:
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree  # Return the class label if it's a leaf node

    root_feature = next(iter(tree))  # Get the root feature
    feature_value = sample[root_feature]

    if feature_value in tree[root_feature]:
        return predict(tree[root_feature][feature_value], sample)
    else:
        return 0  # Default class if feature value is missing


In [ ]:
test_sample = np.array([1, 1, 0, 0])  # Outlook=Overcast, Temperature=Mild, Humidity=High, Wind=Weak
prediction = predict(tree, test_sample)
print(f"Prediction for {test_sample}: {'Play' if prediction == 1 else 'Do Not Play'}")


Prediction for [1 1 0 0]: Play
